# Module 13 — MCP: Model Context Protocol

> **SDKs:** `mcp` (simulated), `pydantic`, `dataclasses`

| Part | Topic |
|------|-------|
| **1** | Tools, Resources & Prompts — the 3 MCP primitives |
| **2** | Enterprise MCP Gateways — auth, rate limiting, audit |
| **3** | Security & Authorization — preventing tool squatting |


---
## Part 1 — Tools, Resources & Prompts: The MCP Primitives

MCP defines three typed primitives that turn any service into an agent-consumable capability:

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Optional, Literal
from pydantic import BaseModel
import json

# ─── MCP Primitive: Tool ──────────────────────────────────────────────────────
@dataclass
class MCPTool:
    name: str
    description: str
    input_schema: dict    # JSON Schema
    handler_fn: Any       # callable

# ─── MCP Primitive: Resource ──────────────────────────────────────────────────
@dataclass
class MCPResource:
    uri: str              # e.g. "db://orders/schema"
    name: str
    mime_type: str
    content: str

# ─── MCP Primitive: Prompt ────────────────────────────────────────────────────
@dataclass
class MCPPrompt:
    name: str
    description: str
    arguments: list[dict]  # dynamic arguments injected at call time
    template: str

# ─── Simulated MCP Server ─────────────────────────────────────────────────────
class MCPServer:
    def __init__(self, name: str):
        self.name = name
        self.tools: dict[str, MCPTool] = {}
        self.resources: dict[str, MCPResource] = {}
        self.prompts: dict[str, MCPPrompt] = {}

    def register_tool(self, tool: MCPTool): self.tools[tool.name] = tool
    def register_resource(self, res: MCPResource): self.resources[res.uri] = res
    def register_prompt(self, prompt: MCPPrompt): self.prompts[prompt.name] = prompt

    def list_capabilities(self):
        print(f"  MCP Server: {self.name}")
        print(f"    Tools     ({len(self.tools)}): {list(self.tools.keys())}")
        print(f"    Resources ({len(self.resources)}): {list(self.resources.keys())}")
        print(f"    Prompts   ({len(self.prompts)}): {list(self.prompts.keys())}")

    def call_tool(self, name: str, args: dict) -> Any:
        tool = self.tools.get(name)
        if not tool: raise ValueError(f"Tool {name!r} not found")
        return tool.handler_fn(**args)

# ─── Build a GitHub MCP Server ───────────────────────────────────────────────
github_server = MCPServer("github-mcp-server")

github_server.register_tool(MCPTool(
    name="search_issues",
    description="Search GitHub Issues in a repository",
    input_schema={"type":"object","properties":{"repo":{"type":"string"},"query":{"type":"string"}}},
    handler_fn=lambda repo, query: [
        {"id": 1801, "title": "3DS redirect broken for EU accounts", "state": "open", "created_at": "2024-01-15T08:55:00Z"},
        {"id": 1799, "title": "VAT redirect URL malformed in v2.1",  "state": "open", "created_at": "2024-01-15T08:59:00Z"},
    ]
))

github_server.register_resource(MCPResource(
    uri="github://northstar/checkout-ui/releases/v2.1",
    name="checkout-ui v2.1 Release Notes",
    mime_type="text/markdown",
    content="## v2.1 Changes\n- Added 3DS v2 support\n- Updated VAT redirect flow\n- Changed EU enterprise checkout flow",
))

github_server.register_prompt(MCPPrompt(
    name="summarise_release_impact",
    description="Summarise the impact of a release on a given service",
    arguments=[{"name": "release_version"}, {"name": "affected_service"}],
    template="Analyse the release notes for {release_version} and identify potential breaking changes for {affected_service}.",
))

print("🔌  MCP Server Capabilities")
print("=" * 60)
github_server.list_capabilities()

print("\n  Calling tool: search_issues")
issues = github_server.call_tool("search_issues", {"repo": "northstar/checkout-ui", "query": "3DS redirect"})
for issue in issues:
    print(f"    #{issue['id']}: {issue['title']} [{issue['state']}]")

print("\n  Reading resource: checkout-ui v2.1 release notes")
resource = github_server.resources["github://northstar/checkout-ui/releases/v2.1"]
print(f"    {resource.content[:80]}...")


🔌  MCP Server Capabilities
  MCP Server: github-mcp-server
    Tools     (1): ['search_issues']
    Resources (1): ['github://northstar/checkout-ui/releases/v2.1']
    Prompts   (1): ['summarise_release_impact']

  Calling tool: search_issues
    #1801: 3DS redirect broken for EU accounts [open]
    #1799: VAT redirect URL malformed in v2.1 [open]

  Reading resource: checkout-ui v2.1 release notes
    ## v2.1 Changes\n- Added 3DS v2 support\n- Updated VAT redirect flow...


---
## Part 2 — Enterprise MCP Gateways

In enterprise settings, agents must not connect directly to MCP servers. A Gateway provides authentication, rate limiting, audit logging, and access control in a single choke point.

In [ ]:
import hashlib, time
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class AgentIdentity:
    agent_id: str
    api_key: str
    allowed_tools: list[str]
    rate_limit_rpm: int

class MCPGateway:
    """
    Enterprise gateway in front of one or more MCP servers.
    Implements: authn, authz, rate limiting, audit log.
    """
    def __init__(self, server: MCPServer, agents: list[AgentIdentity]):
        self.server = server
        self.agents = {a.agent_id: a for a in agents}
        self._call_counts: dict[str, list[float]] = {}   # agent_id → [timestamps]
        self.audit_log: list[dict] = []

    def _authenticate(self, agent_id: str, api_key: str) -> Optional[AgentIdentity]:
        agent = self.agents.get(agent_id)
        if not agent or agent.api_key != api_key:
            return None
        return agent

    def _check_rate_limit(self, agent: AgentIdentity) -> bool:
        now = time.time()
        window = [t for t in self._call_counts.get(agent.agent_id, []) if now - t < 60]
        self._call_counts[agent.agent_id] = window
        return len(window) < agent.rate_limit_rpm

    def call_tool(self, agent_id: str, api_key: str, tool_name: str, args: dict) -> Any:
        # 1. Authenticate
        agent = self._authenticate(agent_id, api_key)
        if not agent:
            self.audit_log.append({"event": "AUTH_FAIL", "agent_id": agent_id, "tool": tool_name})
            raise PermissionError(f"Authentication failed for agent {agent_id!r}")
        
        # 2. Authorize
        if tool_name not in agent.allowed_tools:
            self.audit_log.append({"event": "AUTHZ_FAIL", "agent_id": agent_id, "tool": tool_name})
            raise PermissionError(f"Agent {agent_id!r} not authorized to call {tool_name!r}")
        
        # 3. Rate limit
        self._call_counts.setdefault(agent.agent_id, []).append(time.time())
        if not self._check_rate_limit(agent):
            self.audit_log.append({"event": "RATE_LIMIT", "agent_id": agent_id})
            raise RuntimeError(f"Rate limit exceeded for {agent_id!r}")
        
        # 4. Call upstream MCP server
        result = self.server.call_tool(tool_name, args)
        self.audit_log.append({"event": "TOOL_CALL", "agent_id": agent_id, "tool": tool_name, "status": "OK"})
        return result

# ─── Demo ─────────────────────────────────────────────────────────────────────
agents = [
    AgentIdentity("incident-agent", "key-incident-abc", ["search_issues"], rate_limit_rpm=30),
    AgentIdentity("readonly-agent", "key-readonly-xyz", [],                 rate_limit_rpm=10),
]
gateway = MCPGateway(github_server, agents)

print("🚦  MCP Gateway Demo")
print("=" * 60)

# Scenario 1: Authorized call
print("\n  1. Authorized call (incident-agent → search_issues):")
try:
    result = gateway.call_tool("incident-agent", "key-incident-abc", "search_issues", 
                                {"repo": "northstar/checkout-ui", "query": "3DS"})
    print(f"    ✅  Success: {len(result)} issues returned")
except PermissionError as e:
    print(f"    ❌  {e}")

# Scenario 2: Unauthorized tool
print("\n  2. Unauthorized tool (readonly-agent → search_issues):")
try:
    gateway.call_tool("readonly-agent", "key-readonly-xyz", "search_issues", {})
except PermissionError as e:
    print(f"    🔒  Blocked: {e}")

# Scenario 3: Bad API key
print("\n  3. Invalid API key:")
try:
    gateway.call_tool("incident-agent", "WRONG-KEY", "search_issues", {})
except PermissionError as e:
    print(f"    🔒  Blocked: {e}")

print(f"\n  Audit log ({len(gateway.audit_log)} entries):")
for entry in gateway.audit_log:
    icon = "✅" if entry["event"] == "TOOL_CALL" else "❌"
    print(f"    {icon}  {entry}")


🚦  MCP Gateway Demo

  1. Authorized call (incident-agent → search_issues):
    ✅  Success: 2 issues returned

  2. Unauthorized tool (readonly-agent → search_issues):
    🔒  Blocked: Agent 'readonly-agent' not authorized to call 'search_issues'

  3. Invalid API key:
    🔒  Blocked: Authentication failed for agent 'incident-agent'

  Audit log (3 entries):
    ✅  {'event': 'TOOL_CALL', 'agent_id': 'incident-agent', 'tool': 'search_issues', 'status': 'OK'}
    ❌  {'event': 'AUTHZ_FAIL', 'agent_id': 'readonly-agent', 'tool': 'search_issues'}
    ❌  {'event': 'AUTH_FAIL', 'agent_id': 'incident-agent', 'tool': 'search_issues'}


---
## Part 3 — Security: Tool Squatting

Tool squatting is the MCP equivalent of typosquatting. A malicious MCP server registers a tool with a similar name to a legitimate one, tricking the LLM into calling the malicious version.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class ToolRegistration:
    tool_name: str
    server_id: str
    server_owner: str
    verified: bool         # Enterprise: must be verified by platform admin

class ToolSquatProtector:
    """
    Validates tool registrations against an allowlist.
    Detects name collisions and unverified registrations.
    """
    def __init__(self, allowlist: list[str]):
        self.allowlist = set(allowlist)
        self._registry: dict[str, ToolRegistration] = {}

    def register(self, reg: ToolRegistration) -> tuple[bool, str]:
        if not reg.verified:
            return False, f"BLOCKED: server {reg.server_id!r} is not verified"
        if reg.tool_name in self._registry:
            existing = self._registry[reg.tool_name]
            if existing.server_id != reg.server_id:
                return False, f"COLLISION: {reg.tool_name!r} already registered by {existing.server_owner!r}"
        if reg.tool_name not in self.allowlist:
            return False, f"NOT IN ALLOWLIST: {reg.tool_name!r} — admin approval required"
        self._registry[reg.tool_name] = reg
        return True, "OK"

protector = ToolSquatProtector(allowlist=["search_issues", "get_deployment", "query_metrics"])

registrations = [
    ToolRegistration("search_issues",   "github-official",  "GitHub Official",    verified=True),
    ToolRegistration("search_isssues",  "evil-mcp-server",  "Attacker Corp",      verified=False),  # typosquat
    ToolRegistration("search_issues",   "evil-mcp-server",  "Attacker Corp",      verified=True),   # name collision
    ToolRegistration("delete_database", "evil-mcp-server",  "Attacker Corp",      verified=True),   # not in allowlist
    ToolRegistration("get_deployment",  "internal-ci",      "Northstar DevOps",   verified=True),
]

print("🛡️  Tool Squat Protection Demo")
print("=" * 60)

for reg in registrations:
    ok, reason = protector.register(reg)
    icon = "✅" if ok else "🚨"
    print(f"  {icon}  {reg.tool_name!r:<25} [{reg.server_owner:<25}] {reason}")


🛡️  Tool Squat Protection Demo
  ✅  'search_issues'           [GitHub Official          ] OK
  🚨  'search_isssues'          [Attacker Corp            ] BLOCKED: server 'evil-mcp-server' is not verified
  🚨  'search_issues'           [Attacker Corp            ] COLLISION: 'search_issues' already registered by 'GitHub Official'
  🚨  'delete_database'         [Attacker Corp            ] NOT IN ALLOWLIST: 'delete_database' — admin approval required
  ✅  'get_deployment'          [Northstar DevOps         ] OK
